### Structured Output

Models can be requested to provide their response in a format matching a given schema. This is useful for ensuring the output can be easily parsed and used in subsequent processing. LangChain supports multiple schema types and methods for enforcing structured output.

### Pydantic

Pydantic models provide the richest feature set with field validation, descriptions, and nested structures.

In [1]:
import os
from langchain.chat_models import init_chat_model
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
model

ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001CD0BA82750>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001CD0BBEDE10>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None)

In [2]:
from pydantic import BaseModel, Field

class Movie(BaseModel): 
    title: str = Field(description="The title of the movie")
    year: int = Field(description="This year the movie was released")
    director: str = Field(description="The director of the movie")
    ratings: float = Field(description="The movies ratings out of 10")

In [4]:
model_with_structure = model.with_structured_output(Movie)
model_with_structure

_ChatModelBinding(bound=ChatGroq(output_version=None, profile={'max_input_tokens': 131072, 'max_output_tokens': 16384, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': True, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x000001CD0BA82750>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001CD0BBEDE10>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'Movie', 'description': '', 'parameters': {'properties': {'title': {'description': 'The title of the movie', 'type': 'string'}, 'year': {'description': 'This year the movie was released', 'type': 'integer'}, 'director': {'description': 'The director of the movie', 'type': 'string'}, 'ratings': {'description': 'The movies ratings o

In [5]:
model.invoke("Provide details about the movie inception")

AIMessage(content='<think>\nOkay, so I need to provide details about the movie Inception. Let me start by recalling what I know about it. Directed by Christopher Nolan, right? I think it came out around 2010. The main idea is about dreams within dreams. The lead character is maybe someone like Dom Cobb, played by Leonardo DiCaprio. The concept involves entering people\'s dreams to steal information or plant ideas. There\'s a lot of action and visual effects, especially with the folding city scene. \n\nI should check the plot structure. The movie has multiple layers of dreams, each with different time speeds. The team uses a device called the PAS to enter dreams. There\'s a part where they\'re in a zero-gravity room, maybe during a fight scene. The antagonist wants to steal something, and the team tries to do the opposite, which is planting an idea, known as "inception." \n\nThe cast includes Leonardo DiCaprio as Dom Cobb, Joseph Gordon-Levitt as Arthur, Ellen Page as Ariadne, Tom Hardy

In [6]:
model_with_structure.invoke("Provide details about the movie inception")

Movie(title='Inception', year=2010, director='Christopher Nolan', ratings=8.8)

### Message output alongisde parsed structure

In [7]:
from pydantic import BaseModel, Field

class Movie(BaseModel):
    """A movie with details"""

    title: str = Field(..., description="The title of the movie")
    year: int = Field(..., description="The year the movie was released")
    director: str = Field(..., description="The director of the movie")
    rating: float = Field(..., description="The movie's rating out of 10")

model_with_structure = model.with_structured_output(Movie, include_raw=True)

response = model_with_structure.invoke("Provide details about the move inception")
response

{'raw': AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for details about the movie "Inception." Let me check the tools provided. There\'s a function called Movie that requires title, year, director, and rating. I need to fill in those parameters. I remember that Inception was directed by Christopher Nolan and released in 2010. The rating is probably around 8.8 on IMDb. Let me confirm the exact year and director. Yep, 2010 and Christopher Nolan. The rating is 8.8. So I\'ll structure the tool call with those details.\n', 'tool_calls': [{'id': 'aqdxegf86', 'function': {'arguments': '{"director":"Christopher Nolan","rating":8.8,"title":"Inception","year":2010}', 'name': 'Movie'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 165, 'prompt_tokens': 230, 'total_tokens': 395, 'completion_time': 0.256046013, 'completion_tokens_details': {'reasoning_tokens': 117}, 'prompt_time': 0.009340147, 'prompt_tokens_details': None, 

### Nested Structure

In [8]:
from pydantic import BaseModel, Field

class Actor(BaseModel):
    name: str
    role: str

class MovieDetails(BaseModel):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie inception")
response

MovieDetails(title='Inception', year=2010, cast=[Actor(name='Leonardo DiCaprio', role='Dom Cobb'), Actor(name='Joseph Gordon-Levitt', role='Arthur'), Actor(name='Ellen Page', role='Ariadne'), Actor(name='Tom Hardy', role='Bane')], genres=['Science Fiction', 'Action', 'Thriller'], budget=160.0)

### TypedDict

TypeDict provides a similar alternative using Python's built-in typing, ideal when you don't need runtime validation

In [9]:
from typing_extensions import TypedDict, Annotated

class MovieDict(TypedDict):
    """A movie with details."""

    title: Annotated[str, ..., "The title of the movie"]
    year: Annotated[int, ..., "The year the movie was released"]
    director: Annotated[str, ..., "The director of the movie"]
    rating: Annotated[float, ..., "The movie's rating out of 10"]

model_withTypeDict = model.with_structured_output(MovieDict)
response = model_withTypeDict.invoke("Please provide the details of the movie Avengers")
response

{'director': 'Joss Whedon', 'rating': 8, 'title': 'Avengers', 'year': 2012}

In [10]:
class Actor(TypedDict):
    name: str
    role: str

class MovieDetails(TypedDict):
    title: str
    year: int
    cast: list[Actor]
    genres: list[str]
    budget: float | None = Field(None, description="Budget in millions USD")

model_with_structure = model.with_structured_output(MovieDetails)

response = model_with_structure.invoke("Provide details about the movie inception")
response

{'budget': 160000000,
 'cast': [{'name': 'Leonardo DiCaprio', 'role': 'Dom Cobb'},
  {'name': 'Joseph Gordon-Levitt', 'role': 'Arthur'},
  {'name': 'Ellen Page', 'role': 'Ariadne'}],
 'genres': ['Science Fiction', 'Action'],
 'title': 'Inception',
 'year': 2010}

In [11]:
model.profile

{'max_input_tokens': 131072,
 'max_output_tokens': 16384,
 'image_inputs': False,
 'audio_inputs': False,
 'video_inputs': False,
 'image_outputs': False,
 'audio_outputs': False,
 'video_outputs': False,
 'reasoning_output': True,
 'tool_calling': True}

### DataClasses

A data class is a class typically containing mainly data, although there aren’t really any restrictions. You create it using the `@dataclass` decorator.

In [12]:
import os
os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

In [13]:
from pydantic import BaseModel, Field
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent


class ContactInfo(BaseModel):
    """Contact information for a person."""

    name: str = Field(description="The name of the person")
    email: str = Field(description="The email address of the person")
    phone: str = Field(description="The phone number of the person")


# Initialize Groq model
model = init_chat_model(
    "qwen/qwen3-32b",
    model_provider="groq"
)

# Create agent with structured output
agent = create_agent(
    model=model,
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
        }
    ]
})

print(result["structured_response"])

# ContactInfo(
#     name='John Doe',
#     email='john@example.com',
#     phone='(555) 123-4567'
# )

name='John Doe' email='john@example.com' phone='(555) 123-4567'


In [14]:
from typing_extensions import TypedDict, Annotated
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent


class ContactInfo(TypedDict):
    """Contact information for a person."""

    name: Annotated[str, ..., "The name of the person"]
    email: Annotated[str, ..., "The email address of the person"]
    phone: Annotated[str, ..., "The phone number of the person"]


# Initialize Groq model
model = init_chat_model(
    "qwen/qwen3-32b",
    model_provider="groq"
)

# Create agent with structured output
agent = create_agent(
    model=model,
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
        }
    ]
})

print(result["structured_response"])

# {
#     'name': 'John Doe',
#     'email': 'john@example.com',
#     'phone': '(555) 123-4567'
# }

{'name': 'John Doe', 'email': 'john@example.com', 'phone': '(555) 123-4567'}


In [16]:
### DataClass

from dataclasses import dataclass
from langchain.chat_models import init_chat_model
from langchain.agents import create_agent


@dataclass
class ContactInfo:
    """Contact information for a person."""

    name: str   # The name of the person
    email: str  # The email address of the person
    phone: str  # The phone number of the person


# Initialize Groq model
model = init_chat_model(
    "qwen/qwen3-32b",
    model_provider="groq"
)

# Create agent with structured output
agent = create_agent(
    model=model,
    response_format=ContactInfo
)

result = agent.invoke({
    "messages": [
        {
            "role": "user",
            "content": "Extract contact info from: John Doe, john@example.com, (555) 123-4567"
        }
    ]
})

print(result["structured_response"])

# ContactInfo(
#     name='John Doe',
#     email='john@example.com',
#     phone='(555) 123-4567'
# )

ContactInfo(name='John Doe', email='john@example.com', phone='(555) 123-4567')
